In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
docs = [
    "The cat sat on the mat",
    "The dog sat on the log",
    "The cat and the dog are friends",
    "Dogs and cats are pets",
]
for i, d in enumerate(docs): print(f"D{i+1}: {d}")

In [ ]:
def tokenize(t):
    return re.findall(r"(?u)\b\w\w+\b", t.lower())

vocab = sorted({w for d in docs for w in tokenize(d)})
word2idx = {w: i for i, w in enumerate(vocab)}

bow = np.zeros((len(docs), len(vocab)), dtype=int)
for i, d in enumerate(docs):
    for w in tokenize(d):
        bow[i, word2idx[w]] += 1

df_manual = pd.DataFrame(bow, columns=vocab, index=[f"D{i+1}" for i in range(len(docs))])
print("Vocabulary size:", len(vocab))
df_manual

In [ ]:
cv = CountVectorizer()
X = cv.fit_transform(docs)
df_sk = pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out(), index=df_manual.index)
print("Manual BoW == sklearn BoW :", np.array_equal(df_manual.values, df_sk.values))
df_sk

In [ ]:
print("Binary BoW:")
display(pd.DataFrame(CountVectorizer(binary=True).fit_transform(docs).toarray(),
                     columns=cv.get_feature_names_out()))

cv_stop = CountVectorizer(stop_words="english")
print("\nAfter stop-word removal:", cv_stop.fit(docs).get_feature_names_out())

cv_bi = CountVectorizer(ngram_range=(1, 2))
cv_bi.fit(docs)
print("\nUnigrams + bigrams vocabulary size:", len(cv_bi.vocabulary_))

In [ ]:
import matplotlib.pyplot as plt
new = ["The cat chased the dog"]
print(dict(zip(cv.get_feature_names_out(), cv.transform(new).toarray()[0].tolist())))

plt.figure(figsize=(8, 3))
plt.imshow(df_sk.values, cmap="Blues")
plt.xticks(range(len(vocab)), vocab, rotation=60); plt.yticks(range(len(docs)), df_sk.index)
plt.colorbar(label="count"); plt.title("Document-Term Matrix (BoW)"); plt.show()